<a href="https://colab.research.google.com/github/sabriddinovna/graph_nn/blob/main/graph_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import torch
os.environ['TORCH']=torch.__version__
print(torch.__version__)

2.11.0+cpu


In [5]:
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.1/682.1 kB 17.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


The most common task for graph classification is molecular property prediction, in which molecules are represented as graphs, and the task may be to infer whether a molecule inhibits HIV virus replication or not.

In [6]:
import torch
from torch_geometric.datasets import TUDataset

dataset=TUDataset(root='data/TUDataset', name='MUTAG')


Processing...
Done!


In [7]:
data=dataset[0]

In [ ]:
len(data.x)

17

In [ ]:
data.num_nodes

17

In [ ]:
data.num_edges

38

In [ ]:
len(data.y)

1

In [ ]:
dataset.num_features

7

In [ ]:
len(dataset)

188

In [8]:
torch.manual_seed(32423)
dataset=dataset.shuffle()

train_data=dataset[:150]
test_data=dataset[150:]

In [10]:
from torch_geometric.loader import DataLoader
train_loader=DataLoader(train_data,  batch_size=50, shuffle=True)
test_loader=DataLoader(test_data, batch_size=50, shuffle=False)

for step,  data in enumerate(train_loader):
  print(step+1)
  print(data.num_graphs)
  print(data)



1
50
DataBatch(edge_index=[2, 2022], x=[913, 7], edge_attr=[2022, 4], y=[50], batch=[913], ptr=[51])
2
50
DataBatch(edge_index=[2, 1950], x=[885, 7], edge_attr=[1950, 4], y=[50], batch=[885], ptr=[51])
3
50
DataBatch(edge_index=[2, 1930], x=[877, 7], edge_attr=[1930, 4], y=[50], batch=[877], ptr=[51])


In [12]:
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool

In [17]:
class GCN(torch.nn.Module):
  def __init__(self, hidden_channels):
    super(GCN, self).__init__()

    torch.manual_seed(2334)
    self.conv1=GCNConv(dataset.num_node_features, hidden_channels)
    self.conv2=GCNConv(hidden_channels, hidden_channels)
    self.conv3=GCNConv(hidden_channels, hidden_channels)
    self.lin=Linear(hidden_channels, dataset.num_classes)


  def forward(self,x,edge_index,batch):
    x=self.conv1(x, edge_index)
    x=x.relu()
    x=self.conv2(x, edge_index)
    x=x.relu()
    x=self.conv3(x, edge_index)
    x=global_mean_pool(x, batch)
    x=F.dropout(x,p=0.5,training=self.training)
    x=self.lin(x)

    return x


In [18]:
model=GCN(hidden_channels=50)
print(model)

GCN(
  (conv1): GCNConv(7, 50)
  (conv2): GCNConv(50, 50)
  (conv3): GCNConv(50, 50)
  (lin): Linear(in_features=50, out_features=2, bias=True)
)


In [19]:
from IPython.display import Javascript
display(Javascript('''google.colab.output.setIframeHeight(0, true, {maxHeight: 300})'''))


<IPython.core.display.Javascript object>

In [20]:
model=GCN(hidden_channels=50)
optimizer=torch.optim.Adam(model.parameters(), lr=0.01)
criterion=torch.nn.CrossEntropyLoss()

In [21]:
def train():
  model.train()

  for data in train_loader:
    out=model(data.x, data.edge_index, data.batch)
    loss=criterion(out, data.y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

def test(loader):
  model.eval()

  correct=0
  for data in loader:
    out=model(data.x, data.edge_index, data.batch)
    pred=out.argmax(dim=1)
    correct=correct+int((pred==data.y).sum())

  return correct/len(loader.dataset)

for epoch in range(1, 171, 15):

  train()
  train_acc=test(train_loader)
  test_acc=test(test_loader)
  print(f'Epoch: {epoch:03d}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')

Epoch: 001, Train Acc: 0.6667, Test Acc: 0.6579
Epoch: 016, Train Acc: 0.6667, Test Acc: 0.6579
Epoch: 031, Train Acc: 0.6667, Test Acc: 0.6579
Epoch: 046, Train Acc: 0.6667, Test Acc: 0.6579
Epoch: 061, Train Acc: 0.6667, Test Acc: 0.6579
Epoch: 076, Train Acc: 0.6667, Test Acc: 0.6579
Epoch: 091, Train Acc: 0.6667, Test Acc: 0.6579
Epoch: 106, Train Acc: 0.6933, Test Acc: 0.6579
Epoch: 121, Train Acc: 0.7467, Test Acc: 0.6316
Epoch: 136, Train Acc: 0.7467, Test Acc: 0.6316
Epoch: 151, Train Acc: 0.7467, Test Acc: 0.6842
Epoch: 166, Train Acc: 0.7333, Test Acc: 0.6579


In [22]:
import os
import torch
os.environ['TORCH']=torch.__version__
print(torch.__version__)

2.11.0+cpu


In [23]:

!pip install pyg-lib -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install git+https://github.com/pyg-team/pytorch_geometric.git

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.8 MB/s eta 0:00:00
  Cloning https://github.com/pyg-team/pytorch_geometric.git to /tmp/pip-req-build-7clbma9d
  Running command git clone --filter=blob:none --quiet https://github.com/pyg-team/pytorch_geometric.git /tmp/pip-req-build-7clbma9d
  Resolved https://github.com/pyg-team/pytorch_geometric.git to commit 79d33965a40b7fa83616a9f598a0f8619f25d939
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
